In [20]:
import pandas as pd
df = pd.read_parquet("../dataset.parquet")

In [16]:
# df.head()
# df.info()
df.describe(include='all')

,Unique Key,Created Date,Closed Date,Agency,Agency Name,Complaint Type,Descriptor,Location Type,Incident Zip,Incident Address,...,Park Borough,Vehicle Type,Taxi Company Borough,Taxi Pick Up Location,Bridge Highway Name,Bridge Highway Direction,Road Ramp,Bridge Highway Segment,Latitude,Longitude
count,2449109.0,2449109,2449109,2449109,2449109,2449109,2449109,2449109,2449109,2449109,...,2449109,120172,742,15298,7458,2127,1642,7458,2449109.0,2449109.0
unique,<NA>,NaN,NaN,15,15,169,867,146,244,473530,...,6,9,5,9028,55,196,349,146,<NA>,<NA>
top,<NA>,NaN,NaN,NYPD,NEW YORK CITY POLICE DEPARTMENT,ILLEGAL PARKING,LOUD MUSIC/PARTY,STREET/SIDEWALK,10466,UNKNOWN,...,BROOKLYN,Car,MANHATTAN,"JOHN F KENNEDY AIRPORT, QUEENS (JAMAICA) ,NY, ...",1,1 Downtown,Roadway,Platform,<NA>,<NA>
freq,<NA>,NaN,NaN,1277327,1277327,435694,355812,871293,82495,78649,...,734902,77461,228,1339,695,141,161,2505,<NA>,<NA>
mean,64939171.515552,2025-05-13 13:52:39.768943360,2025-05-18 22:00:12.625519360,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.738062,-73.920616
min,63572086.0,2025-01-01 00:47:49,2025-01-01 00:48:57,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.498913,-74.254952
25%,64248610.0,2025-03-03 18:53:18,2025-03-10 13:50:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.675376,-73.965031
50%,64929063.0,2025-05-12 16:08:22,2025-05-19 08:12:01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.729404,-73.923476
75%,65625320.0,2025-07-21 16:29:30,2025-07-27 10:23:17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.818152,-73.8697
max,66353310.0,2025-10-03 01:19:28,2025-10-03 02:22:24,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.912869,-73.700384


In [21]:
# Convert text to dates
date_fields = ['Created Date', 'Closed Date', 'Due Date']
df[date_fields] = df[date_fields].apply(pd.to_datetime)


/tmp/ipykernel_44175/2864774900.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[date_fields] = df[date_fields].apply(pd.to_datetime)


In [24]:
#Na Data
na = df.isna().sum().sort_values(ascending=False)
na_pct = (na / len(df)) * 100

na_df = pd.DataFrame({
    "Missing Values": na,
    "% of Total": na_pct
})
na_df.head(50)

,Missing Values,% of Total
Unique Key,0,0.0
Created Date,0,0.0
Closed Date,0,0.0
Agency,0,0.0
Agency Name,0,0.0
Complaint Type,0,0.0
Descriptor,0,0.0
Location Type,0,0.0
Incident Zip,0,0.0
Incident Address,0,0.0


In [23]:
# Removing data with more than 30% of missing data
# df = df.loc[:, df.isna().mean() < 0.3]
categorical_columns = ['Agency', 'Agency Name', 'Address Type', 'Status', 'City', 'Resolution Description', 'Community Board' ,'Complaint Type', 'Descriptor', 'Location Type', 'Incident Zip', 'Borough', 'Open Data Channel Type', 'Park Borough']
# Trim the categorical data and make it uppercase for consistency
df[categorical_columns] = df[categorical_columns].apply(lambda x: x.str.strip().str.upper())
# Since closed date is important, remove the rows which are closed but doesnt have a closing date
df.drop(
    df[(df['Status'] == 'CLOSED') & (df['Closed Date'].isna())].index,
    inplace=True
)
# For missing values of city, we try top derive it from borough since it is dependent on borough. Get the mostly mapped city to borough and use that value
city_mapping = (
    df.groupby('Borough')['City']
      .agg(lambda x: x.value_counts().index[0] if x.notna().any() else 'UNKNOWN')
      .to_dict()
)
df['City'] = df.apply(
    lambda row: city_mapping.get(row['Borough'], 'UNKNOWN') if pd.isna(row['City']) else row['City'],
    axis=1
)
# Columns that serve not much info
cols_to_be_dropped = ['Taxi Company Borough', 'Road Ramp', 'Bridge Highway Direction', 'Bridge Highway Name', 'Bridge Highway Segment','Facility Type','Due Date', 'Taxi Pick Up Location','Vehicle Type','Landmark', 'Cross Street 1', 'BBL', 'Cross Street 2', 'Resolution Description', 'Intersection Street 1', 'Intersection Street 2', 'Resolution Action Updated Date', 'Park Facility Name', 'Location']
df = df.drop(columns = cols_to_be_dropped)
# These columns fill missing values with UNKNOWN category
fill_unknown = ['Location Type', 'Street Name', 'Incident Address', 'Descriptor']
for col in fill_unknown:
    df[col] = df[col].fillna('UNKNOWN')
# Address Type has only 0.4% data missing and most of them is skewed to the value "ADDRESS", so simply using the mode
df['Address Type'] = df['Address Type'].fillna(df['Address Type'].mode()[0])

# 4. Drop rows for missing coordinates since these are the important features (tiny percentage)
# TODO: Closed Date, we can keep it for clustering but for supervised learning, this is a key feature
df = df.dropna(subset=['Closed Date', 'Latitude', 'Longitude', 'X Coordinate (State Plane)', 'Y Coordinate (State Plane)'])

# Filling missing ZIP based on nearby points
df['Incident Zip'] = df.groupby('Borough')['Incident Zip'].transform(
    lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else x)
)

# There are columns where created date are after the Clsoed date. Remove these data
df = df[df['Closed Date'] >= df['Created Date']]

In [27]:
df.to_parquet("../cleaned_data.parquet")

In [7]:
df.groupby("Address Type").size().reset_index(name='count').sort_values('count', ascending=False)

,Address Type,count
0,ADDRESS,2352647
2,INTERSECTION,182136
1,BLOCKFACE,31161
3,PLACE,9831
5,UNRECOGNIZED,8057
4,PLACENAME,1533


In [5]:
# show all columns in DataFrame output
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# optionally, avoid truncating text values
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 2000)


In [78]:
pd.reset_option('display.max_columns')
pd.reset_option('display.max_rows')
pd.reset_option('display.max_colwidth')
pd.reset_option('display.width')
